# Compacting the State Space

This notebook accompanies the section *Compacting the State Space* of the lecture notes.  A canonical LR(1) parser
keeps apart states that only differ in the follow tokens of their marked rules.  Such states have the same *core*,
i.e. they contain the same marked rules if we drop the follow tokens.  The state space of the parser can be compacted
by combining the states that have the same core.  <span style="font-variant:small-caps;">Larkita</span> does this if it is called with the keyword argument
`merge_states=True`, but it does not combine two states if this would result in a conflict.  We use the grammar from
the *Dragon Book* that is discussed in the lecture notes in order to show how the state space is compacted.

**Contents**

1. [The Grammar](#The-Grammar)
2. [The Canonical LR(1) Parser](#The-Canonical-LR(1)-Parser)
   - [The Function `states`](#The-Function-states)
   - [The Function `core`](#The-Function-core)
3. [Combining the States with the Same Core](#Combining-the-States-with-the-Same-Core)
4. [Comparing the Parsers](#Comparing-the-Parsers)

## The Grammar

The file `Examples/dragon-book.g` contains the grammar from the *Dragon Book*.  The notebook
`05-LR-Table-Generator.ipynb` uses the same file.

In [ ]:
from IPython.display import Code

In [ ]:
Code(filename='Examples/dragon-book.g', language='larkita')

## The Canonical LR(1) Parser

By default, <span style="font-variant:small-caps;">Larkita</span> computes the canonical LR(1) parser of a grammar.  The keyword argument `out_file` specifies
the file into which <span style="font-variant:small-caps;">Larkita</span> writes the rules and the states of the parser.

In [ ]:
from larkita import Larkita

In [ ]:
canonical = Larkita.open('Examples/dragon-book.g', out_file='dragon-book.out')

The canonical parser has 10 states.  These are the states of the LR-goto-graph shown in the lecture notes, and
they are numbered in the same way.

In [ ]:
len(canonical.parse_table.states)

### The Function `states`

In order to find the states that have the same core, we read the marked rules of every state from the output file.
In the section `States` of this file, every state starts with a line of the form `State n`.  It is followed by its
marked rules, which are the only lines of the state that contain the marker `•`, and then by its actions.  Every
marked rule is followed by a colon and its follow tokens, e.g.
```
(2)  c → "x" • c:  "x", "y"
```
The closure rules are marked with a `+` at the beginning of the line.  We drop this marker, since it is irrelevant
for the core.

**Function `states(report)`**
- *Input:* `report` is the contents of an output file of Larkita.
- *Output:* A dictionary that maps the number of every state to the list of its marked rules.  Every marked rule is
  given as a string that contains the rule, the marker, and the follow tokens.

In [ ]:
def states(report: str) -> dict[int, list[str]]:
    Result: dict[int, list[str]] = {}
    for block in report[report.index('\nStates\n'):].split('\n\n\nState ')[1:]:
        lines = block.split('\n')
        Result[int(lines[0].split()[0])] = [line.strip().lstrip('+').strip() for line in lines if '•' in line]
    return Result

**Function `print_state(States, n)`**
- *Input:* `States` is a dictionary as returned by the function `states` and `n` is the number of a state.
- *Output:* None.  The function prints the marked rules of the state `n`.

In [ ]:
def print_state(States: dict[int, list[str]], n: int) -> None:
    print(f'State {n}')
    for marked_rule in States[n]:
        print('    ' + marked_rule)

In [ ]:
with open('dragon-book.out', encoding='utf-8') as handle:
    CanonicalStates = states(handle.read())

The states 3 and 6 differ only in their follow tokens:  In state 3, every marked rule has the follow tokens `"x"`
and `"y"`, while in state 6, every marked rule has the follow token `$END`, which marks the end of the input.

In [ ]:
print_state(CanonicalStates, 3)
print_state(CanonicalStates, 6)

### The Function `core`

The *core* of a state is the set of its marked rules without their follow tokens.  The follow tokens are separated
from the marked rule by a colon that is followed by at least two blanks.  A colon inside a marked rule, e.g. in the
literal `":="`, is followed by a double quote instead.

In [ ]:
import re

**Function `core(marked_rules)`**
- *Input:* `marked_rules` is the list of the marked rules of a state, as returned by the function `states`.
- *Output:* The set of these marked rules without their follow tokens.

In [ ]:
def core(marked_rules: list[str]) -> frozenset[str]:
    return frozenset(re.split(r':\s\s+', marked_rule)[0] for marked_rule in marked_rules)

**Function `same_cores(States)`**
- *Input:* `States` is a dictionary as returned by the function `states`.
- *Output:* The list of all groups of at least two states that have the same core.  Every group is a sorted list of
  state numbers.

In [ ]:
def same_cores(States: dict[int, list[str]]) -> list[list[int]]:
    Groups: dict[frozenset[str], list[int]] = {}
    for n in sorted(States):
        Groups.setdefault(core(States[n]), []).append(n)
    return [group for group in Groups.values() if len(group) > 1]

Three pairs of states have the same core:  the states 3 and 6, the states 4 and 7, and the states 8 and 9.

In [ ]:
same_cores(CanonicalStates)

## Combining the States with the Same Core

With the keyword argument `merge_states=True`, <span style="font-variant:small-caps;">Larkita</span> combines the states that have the same core, as long as
this does not result in a conflict.  For our grammar, no conflict arises, so all three pairs of states are combined
and the parser has only 7 states.

In [ ]:
compacted = Larkita.open('Examples/dragon-book.g', out_file='dragon-book-compacted.out', merge_states=True)

In [ ]:
len(compacted.parse_table.states)

In the output file, the header of every combined state lists the states of the canonical parser that it has been
combined from.  The states are numbered anew, so the combination of the states 8 and 9 is now state 6.

**Function `combined_states(report)`**
- *Input:* `report` is the contents of an output file of Larkita.
- *Output:* The list of the headers of those states that have been combined from several states.

In [ ]:
def combined_states(report: str) -> list[str]:
    return [line for line in report.split('\n') if line.startswith('State ') and 'merged from' in line]

In [ ]:
with open('dragon-book-compacted.out', encoding='utf-8') as handle:
    report = handle.read()
combined_states(report)

The follow tokens of every marked rule of a combined state are the union of the follow tokens of this rule in the
original states.  For example, the state 3 that has been combined from the states 3 and 6 has the follow tokens
`"x"`, `"y"`, and `$END`.

In [ ]:
print_state(states(report), 3)

## Comparing the Parsers

The canonical parser and the compacted parser return the same abstract syntax trees.  The function `compare` parses
a string with both parsers and checks that the results are equal.

**Function `compare(s)`**
- *Input:* `s` is a string that is generated by the grammar.
- *Output:* None.  The function prints the abstract syntax tree that the canonical parser returns for `s` and whether
  the compacted parser returns the same tree.

In [ ]:
def compare(s: str) -> None:
    tree = canonical.parse(s)
    print(f'{s}: {tree}, same tree: {tree == compacted.parse(s)}')

In [ ]:
for s in ['yy', 'xyy', 'yxy', 'xxyxy']:
    compare(s)